# Moorea quadrat-link check

Decides whether real before/after pairs are possible: **PASS** if at least `mlc.gate_min_quadrats`
(100) quadrats appear in 2+ survey years, matched only by IDs in the file names.

1. Look at the real names (cell 2) and make sure `mlc.filename_pattern` in `configs/data.yaml` fits them.
2. Run the rest. If fewer than 95% of names match, fix the pattern first.
3. Write the result and numbers in `docs/DECISIONS.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

from reefwatch.data.ingest_mlc import build_pairs, ingest, link_check, load_l1_map

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = OmegaConf.load(ROOT / "configs/data.yaml")
mlc_dir = ROOT / cfg.paths.mlc_dir
print("pattern:", cfg.mlc.filename_pattern)

In [ ]:
names = sorted(p.name for p in mlc_dir.rglob("*") if p.is_file())
print(len(names), "files")
names[:30]

In [ ]:
points, rep = ingest(
    mlc_dir,
    cfg.mlc.filename_pattern,
    load_l1_map(ROOT / "configs/taxonomy.yaml"),
    tuple(cfg.mlc.image_exts),
    cfg.mlc.annotation_ext,
)
print(f"images {rep.images} | names matched {rep.match_rate:.1%} | annotation files {rep.annotation_files}")
print(f"points {len(points)} | bad annotation lines {rep.bad_lines}")
print("unmatched names:", rep.unmatched_names[:10])
print("missing annotations:", rep.missing_annotations[:10])
print("labels with no L1 mapping:", sorted(rep.unmapped_labels.items(), key=lambda kv: -kv[1])[:15])

## Result

In [ ]:
check = link_check(points, cfg.mlc.gate_min_quadrats) if len(points) else {}
check

## Quadrats by number of years, and the pair table

In [ ]:
if len(points):
    imgs = points.dropna(subset=["quadrat_id"])[["quadrat_id", "survey_year", "Image_name"]].drop_duplicates()
    print(imgs.groupby("quadrat_id")["survey_year"].nunique().value_counts().sort_index())
    pairs = build_pairs(points)
    print(len(pairs), "pairs;", int((pairs.n_in_year > 1).sum()), "come from quadrats with several images in one year (review these)")
    display(pairs.head())